# Data Ingestion with APIs and Webhooks
**Workshop, 120 minutes. No coding needed.** You only run cells (press the play button).
Some cells have one value you can change. Those are marked **Exercise**.

*Data ingestion* means getting data from somewhere else into your own place. Today we do it two ways: **pull** and **push**.

Run the cells from top to bottom.

## 0. Setup
Type your name below. The presenter will tell you the real `CLASS_TOPIC`. Then run the cell.

In [ ]:
import secrets

MY_NAME = "anon"  # @param {type:"string"}
CLASS_TOPIC = "ws-demo-class"  # @param {type:"string"}

# A private topic only you use (section 2). Random, so nobody else has it.
MY_TOPIC = "ws-" + secrets.token_hex(4)
print("Hello", MY_NAME, "| class topic:", CLASS_TOPIC, "| my topic:", MY_TOPIC)

**Helpers.** Just run this cell. You do not need to read it.
It holds small functions we use later. It also holds some sample orders, used if the internet service is down.

In [ ]:
#@title Helpers - just run this cell { display-mode: "form" }
# Nothing to edit here. These small functions are used by later cells.
import json, os, time, uuid, secrets
from datetime import datetime, timezone
import requests
import pandas as pd

NTFY = "https://ntfy.sh"
DUMMY = "https://dummyjson.com"
# Only for testing: pretend ntfy is down.
FORCE_FALLBACK = os.environ.get("FORCE_FALLBACK") == "1"

# Sample data used when ntfy is down or says "too many requests" (HTTP 429).
# Messages are STRINGS, just like real ntfy messages. Note: 2 duplicates, 1 bad message.
def _o(oid, who, pid, qty):
    return json.dumps({"order_id": oid, "customer": who, "product_id": pid,
                       "qty": qty, "ts": "2026-10-09T10:00:00+00:00"})
SAMPLE_ORDERS = [_o("a1", "Ana", 1, 2), _o("a2", "Budi", 11, 1), _o("a3", "Citra", 12, 3),
    _o("a4", "Dewi", 13, 1), _o("a5", "Eko", 14, 2), _o("a6", "Fajar", 15, 1),
    _o("a7", "Gita", 1, 1), _o("a8", "Hadi", 11, 4), _o("a9", "Indra", 12, 1),
    _o("b1", "Joko", 13, 2), _o("b2", "Kiki", 1, 5), _o("b3", "Lina", 14, 1),
    _o("a2", "Budi", 11, 1),            # duplicate of a2
    _o("b1", "Joko", 13, 2),            # duplicate of b1
    "oops this is not json {"]          # malformed message
SAMPLE_EVENTS = [{"id": f"s{i}", "event": "message", "message": m}
                 for i, m in enumerate(SAMPLE_ORDERS)]

def get_json(path, **params):
    """GET from DummyJSON. Returns (response, data)."""
    r = requests.get(DUMMY + path, params=params, timeout=10)
    r.raise_for_status()
    return r, r.json()

def fetch_all_products(limit=50):
    """Loop over every page of /products and return a small DataFrame."""
    rows, skip, total = [], 0, 1
    while skip < total:
        _, data = get_json("/products", skip=skip, limit=limit)
        total = data["total"]
        rows += [{k: p[k] for k in ("id", "title", "category", "price")} for p in data["products"]]
        skip += limit
    return pd.DataFrame(rows)

def publish(topic, payload):
    """POST a dict as JSON text to ntfy. Returns the message id, or None on failure."""
    try:
        if FORCE_FALLBACK:
            raise requests.ConnectionError("FORCE_FALLBACK is on")
        r = requests.post(f"{NTFY}/{topic}", data=json.dumps(payload), timeout=10)
        r.raise_for_status()
        return r.json()["id"]
    except Exception as e:
        print(f"WARNING: could not publish to ntfy ({e}).")
        return None

def new_since(events, last_id):
    """Cursor logic: events AFTER last_id. No cursor (or unknown id) = everything."""
    ids = [e["id"] for e in events]
    return events[ids.index(last_id) + 1:] if last_id in ids else list(events)

def poll(topic, since="all"):
    """Ask ntfy for stored messages after `since`. Falls back to SAMPLE_EVENTS."""
    try:
        if FORCE_FALLBACK:
            raise requests.ConnectionError("FORCE_FALLBACK is on")
        r = requests.get(f"{NTFY}/{topic}/json", params={"poll": 1, "since": since}, timeout=10)
        r.raise_for_status()
        events = [json.loads(line) for line in r.text.splitlines() if line.strip()]
    except Exception as e:
        print(f"WARNING: ntfy not available ({e}). Using SAMPLE_ORDERS instead.")
        return new_since(SAMPLE_EVENTS, None if since == "all" else since)
    return [e for e in events if e.get("event") == "message"]  # skip open/keepalive

def stream(topic, seconds, on_event):
    """Listen live for `seconds`, call on_event(event) for each real message."""
    end = time.time() + seconds
    try:
        if FORCE_FALLBACK:
            raise requests.ConnectionError("FORCE_FALLBACK is on")
        with requests.get(f"{NTFY}/{topic}/json", stream=True, timeout=(10, seconds)) as r:
            r.raise_for_status()
            try:
                for line in r.iter_lines():
                    if line and json.loads(line).get("event") == "message":
                        on_event(json.loads(line))
                    if time.time() > end:
                        break
            except requests.RequestException:
                pass  # quiet topic: read timed out, that is the normal end
    except Exception as e:
        print(f"WARNING: could not stream from ntfy ({e}).")

def parse_order(message):
    """Turn a message STRING into an order dict, or None if it is not a valid order."""
    try:
        o = json.loads(message)
        ok = (isinstance(o["order_id"], str) and isinstance(o["customer"], str)
              and type(o["product_id"]) is int and type(o["qty"]) is int and o["qty"] > 0)
        return {k: o[k] for k in ("order_id", "customer", "product_id", "qty", "ts")} if ok else None
    except (ValueError, KeyError, TypeError):
        return None

def clean_orders(events):
    """events -> (orders DataFrame without duplicates, skipped, rows_before_dedupe)."""
    good = [parse_order(e["message"]) for e in events]
    orders = [o for o in good if o]
    df = pd.DataFrame(orders, columns=["order_id", "customer", "product_id", "qty", "ts"])
    return df.drop_duplicates("order_id").reset_index(drop=True), len(good) - len(orders), len(df)

def enrich(orders, products):
    """Add title and revenue = price * qty to each order."""
    df = orders.merge(products, left_on="product_id", right_on="id", how="left")
    df["revenue"] = df["price"] * df["qty"]
    return df

def top_products(enriched, n=5):
    """Revenue per product, biggest first."""
    return (enriched.groupby("title", as_index=False)["revenue"].sum()
            .sort_values("revenue", ascending=False).head(n).reset_index(drop=True))


## 1. API (pull)
Think of a restaurant. You sit at a table. You ask the waiter for a dish. The kitchen answers. **You start every request.**

An **API** works the same way. Your code asks a server for data. The server replies.
We use DummyJSON, a free fake shop. No account needed.

In [ ]:
# Ask for product number 1
r, data = get_json("/products/1")
print("Status:", r.status_code)   # 200 means OK
print(data)

**Exercise:** change the number `1` to any number from 1 to 194, then run the cell.

In [ ]:
PRODUCT_ID = 1  # @param {type:"integer"}
r, data = get_json(f"/products/{PRODUCT_ID}")
print(r.status_code, data["title"], "- price:", data["price"])

Now we **search**. We ask for products that match the word `phone`.

In [ ]:
r, data = get_json("/products/search", q="phone")
print("Found:", data["total"])
for p in data["products"]:
    print("-", p["title"], p["price"])

The shop has 194 products. The server will not send them all at once. It sends small groups.
Each group is a page. This is called **pagination**.
`limit` = how many items per page. `skip` = how many items to jump over.

In [ ]:
# Page 3 if each page has 5 items: skip 10, take 5 (ids 11 to 15)
r, data = get_json("/products", skip=10, limit=5)
print("ids:", [p["id"] for p in data["products"]], "| total:", data["total"])

Now a loop gets every page and puts all rows in a table (a pandas **DataFrame**).

In [ ]:
products = fetch_all_products()   # loops over all pages
print(len(products), "products")
products.head()

Servers protect themselves. They limit how many requests you can send. This is a **rate limit**.
The server tells you what is left in a header. Please do not spam it.

In [ ]:
r, _ = get_json("/products/1")
print("Limit:", r.headers.get("x-ratelimit-limit"))
print("Remaining:", r.headers.get("x-ratelimit-remaining"))

## 2. Webhook / push
Now the opposite. "Don't call us, we'll call you."
Think of a pizza shop that rings your phone when the pizza is ready. **The sender starts the message.**

Three words:
- **Event**: something that happened (an order was placed).
- **Payload**: the data sent with the event (the order details).
- **Receiver URL**: the web address that gets the event.

A real shop (Midtrans, Shopify) sends a **webhook** to your receiver URL.
Today we use **ntfy.sh** instead. To be honest: ntfy is a push-over-HTTP message service. It is not a real webhook sender. It stands in for one, so we need no account.

In [ ]:
link = f"https://ntfy.sh/{MY_TOPIC}"
print("Open this link in a new tab, then keep it open:")
print(link)

In [ ]:
# Send one sample order to YOUR private topic. Watch it appear in the other tab.
order = {"order_id": "demo1", "customer": MY_NAME, "product_id": 1, "qty": 2}
print("Sent, id:", publish(MY_TOPIC, order))

**Exercise:** change `3` if you like, then run. Watch the tab.

In [ ]:
HOW_MANY = 3  # @param {type:"integer"}
for i in range(HOW_MANY):
    order = {"order_id": f"demo{i+2}", "customer": MY_NAME, "product_id": i + 1, "qty": 1}
    publish(MY_TOPIC, order)
print("Sent", HOW_MANY, "more orders")

## 3. Pull vs push

| | Pull (API) | Push (webhook) |
|---|---|---|
| Who starts? | You ask | The sender sends |
| Freshness | As fresh as your last ask | Almost instant |
| Load | You choose how often | You must be ready any time |
| Missed data | Safe: ask again later | Can be lost if you are offline |
| History | Often kept, you can go back | Usually not kept for long |

**Quiz 1:** You need a report every morning. Pull or push?

<details><summary>Answer</summary>Pull. You choose the time and ask once.</details>

**Quiz 2:** Your site must react the second a payment is made. Pull or push?

<details><summary>Answer</summary>Push. The sender tells you right away. (A pull loop would be slow or wasteful.)</details>

## 4. Role in data engineering
A data pipeline has steps:

**Source → Ingest → Store → Transform → Serve**

**Ingest** is the step where data enters your system. That is today's topic.

- **API = pull.** Batch or incremental. You deal with pagination, cursors and rate limits.
- **Push = events.** Land the raw data first. Expect duplicates and gaps.
- **In production** teams combine both.

## 5. Hands-on mini pipeline
Everyone sends orders to **one shared topic**. Then we collect, clean and analyse them together.

### 5a. Producer
Pick a product (1 to 194) and a quantity. Run the cell to place an order.

In [ ]:
PRODUCT = 1  # @param {type:"integer"}
QTY = 1  # @param {type:"integer"}
last_order = {"order_id": uuid.uuid4().hex[:8], "customer": MY_NAME, "product_id": PRODUCT,
              "qty": QTY, "ts": datetime.now(timezone.utc).isoformat()}
publish(CLASS_TOPIC, last_order)
print("Placed:", last_order)

Oops, the network was slow, so you press send again. Senders often retry. The **same** order arrives **twice**. We call this a duplicate. Keep it in mind for 5d.

In [ ]:
# Oops, resend my last order (same order_id!)
publish(CLASS_TOPIC, last_order)
print("Resent:", last_order["order_id"])

### 5b. Batch consumer
Now we read the orders. A broker can be a little behind, so wait 2 seconds after sending.
We save the id of the last message. This is a **cursor**. On the next run we ask only for messages after it.
This is an **incremental load**. Run this cell many times.

In [ ]:
time.sleep(2)                      # brokers can be slightly behind
if "raw_events" not in globals():  # first run only
    raw_events, last_id = [], None
new = poll(CLASS_TOPIC, last_id or "all")   # only messages after the cursor
raw_events += new
if new:
    last_id = new[-1]["id"]        # move the cursor
print("New messages this run:", len(new), "| total stored:", len(raw_events))

### 5c. Streaming consumer
Instead of asking again and again, we keep the connection open. Messages are printed as they arrive.
Ask a friend to place an order now! It stops after `STREAM_SECONDS`.

In [ ]:
STREAM_SECONDS = 30  # @param {type:"integer"}
def show(event):
    o = parse_order(event["message"])
    print("LIVE:", o if o else "(not an order)")
print("Listening for", STREAM_SECONDS, "seconds...")
stream(CLASS_TOPIC, STREAM_SECONDS, show)
print("Closed.")

### 5d. Clean
Real data is messy. A ntfy `message` is a **string** of JSON, so we convert it. Bad messages are skipped and counted. Duplicates are removed using `order_id`.

In [ ]:
orders, skipped, before = clean_orders(raw_events)
print("Rows before dedupe:", before)
print("Rows after dedupe: ", len(orders))
print("Skipped (not valid orders):", skipped)
orders.head()

### 5e. Enrich and answer
Join the orders with the `products` table from section 1. Then revenue = price × qty.

In [ ]:
enriched = enrich(orders, products)
top = top_products(enriched)
print("Best-seller tonight:", top.iloc[0]["title"] if len(top) else "no orders yet")
top

### 5f. Run it twice
Idempotent means: running twice gives the same result. Let us reload everything and clean again.

In [ ]:
reload = poll(CLASS_TOPIC, "all")                       # full reload
again, _, _ = clean_orders(raw_events + reload)         # old + reloaded
print("Before:", len(orders), "| after reload:", len(again))
assert again.order_id.is_unique, "duplicates found!"
print("OK: no duplicate order_id")

## 6. Go further
- **Postman**: click-to-try any API, no code.
- **Make / n8n**: build webhook receivers with boxes and arrows.
- **dlt**: Python library that loads API data into a database.
- **Airbyte**: ready-made connectors for hundreds of sources.
- **Kafka**: big event broker. A ntfy topic ≈ a Kafka topic. Our cursor ≈ a consumer offset.
- **Debezium / CDC**: turns database changes into events.
- **Airflow**: schedules and monitors pipelines.